# If you want to fill a new database with the given raw_data, run this code

Aktuell ist der Schritt nur für die Prototyping-/PoC-Phase im Python-Notebook vorgesehen und wird später in ein eigenständiges Skript überführt.

In [1]:
import json
import pymongo
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

DATA_RAW_DIR = Path() / ".." / "data" / "raw"

with open(DATA_RAW_DIR / 'raw_data.json') as f:
    data = json.load(f)

In [2]:
import pandas as pd

data_df = pd.DataFrame(data)

In [3]:
data_df.columns

Index(['title', 'ingredients', 'instructions'], dtype='object')

In [4]:
data_df["title"].size

39802

In [5]:
data_df.isnull().sum()

title           280
ingredients     280
instructions    280
dtype: int64

In [6]:
data_df.dropna(inplace=True)

data_df.isnull().sum()

title           0
ingredients     0
instructions    0
dtype: int64

In [7]:
data_df["title"].size

39522

In [8]:
data_df.head(10)

,title,ingredients,instructions
0,Slow Cooker Chicken and Dumplings,"[4 skinless, boneless chicken breast halves AD...","Place the chicken, butter, soup, and onion in ..."
1,Awesome Slow Cooker Pot Roast,[2 (10.75 ounce) cans condensed cream of mushr...,"In a slow cooker, mix cream of mushroom soup, ..."
2,Brown Sugar Meatloaf,"[1/2 cup packed brown sugar ADVERTISEMENT, 1/2...",Preheat oven to 350 degrees F (175 degrees C)....
3,Best Chocolate Chip Cookies,"[1 cup butter, softened ADVERTISEMENT, 1 cup w...",Preheat oven to 350 degrees F (175 degrees C)....
4,Homemade Mac and Cheese Casserole,[8 ounces whole wheat rotini pasta ADVERTISEME...,Preheat oven to 350 degrees F. Line a 2-quart ...
5,Banana Banana Bread,"[2 cups all-purpose flour ADVERTISEMENT, 1 tea...",Preheat oven to 350 degrees F (175 degrees C)....
6,Chef John's Fisherman's Pie,"[For potato crust: ADVERTISEMENT, 3 russet pot...",Bring a large saucepan of salted water and to ...
7,Mom's Zucchini Bread,"[3 cups all-purpose flour ADVERTISEMENT, 1 tea...",Grease and flour two 8 x 4 inch pans. Preheat ...
8,The Best Rolled Sugar Cookies,"[1 1/2 cups butter, softened ADVERTISEMENT, 2 ...","In a large bowl, cream together butter and sug..."
9,Singapore Chili Crabs,"[Sauce: ADVERTISEMENT, 1/2 cup ketchup ADVERTI...","Whisk ketchup, chicken broth, egg, soy sauce, ..."


In [9]:
# remove all ADVERTISEMENT strings from ingredients

data_df["ingredients"] = data_df["ingredients"].apply(lambda x: [i.replace("ADVERTISEMENT", "") for i in x])

In [10]:
data_df.head()

,title,ingredients,instructions
0,Slow Cooker Chicken and Dumplings,"[4 skinless, boneless chicken breast halves , ...","Place the chicken, butter, soup, and onion in ..."
1,Awesome Slow Cooker Pot Roast,[2 (10.75 ounce) cans condensed cream of mushr...,"In a slow cooker, mix cream of mushroom soup, ..."
2,Brown Sugar Meatloaf,"[1/2 cup packed brown sugar , 1/2 cup ketchup ...",Preheat oven to 350 degrees F (175 degrees C)....
3,Best Chocolate Chip Cookies,"[1 cup butter, softened , 1 cup white sugar , ...",Preheat oven to 350 degrees F (175 degrees C)....
4,Homemade Mac and Cheese Casserole,"[8 ounces whole wheat rotini pasta , 3 cups fr...",Preheat oven to 350 degrees F. Line a 2-quart ...


In [11]:
data_df = data_df[:100]

In [12]:
import os
Client = pymongo.MongoClient(os.getenv("MONGO_DB_URL"))
db = Client[os.getenv("MONGO_DB_NAME")]
collection = db[os.getenv("MONGO_COLLECTION_NAME_RECIPES")]
collection.delete_many({})

collection.insert_many(item for item in data_df.to_dict(orient="records"))

ServerSelectionTimeoutError: SSL handshake failed: ac-z5prppd-shard-00-01.jlbl0jj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1018) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-z5prppd-shard-00-02.jlbl0jj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1018) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-z5prppd-shard-00-00.jlbl0jj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1018) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69a7e9be3af11c7c5fdf8d53, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-z5prppd-shard-00-00.jlbl0jj.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-z5prppd